In [13]:
import json
import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# ==========================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==========================================
MODEL_NAME = "xlm-roberta-base"
JSONL_PATH = "./combined_tamil_tanglish.jsonl"
OUTPUT_DIR = "./xlmr_robert_base_model"
LOGS_DIR = "./logs"

MAX_LEN = 384  # Increased to comfortably fit the concatenated text strings
BATCH_SIZE = 8  # Adjust based on your GPU VRAM (e.g., 8 or 16)
EPOCHS = 4
LEARNING_RATE = 2e-5

# All 7 target fields specified by the user
LABEL_COL = "intent_type"
TEXT_FIELDS = [
    "subject_cleaned_2",
    "body_cleaned_2",
    "tamil_subject",
    "tamil_body",
    "tanglish_subject",
    "tanglish_body",
]


In [14]:
# ==========================================
# 2. DATA LOADING AND CLEANING
# ==========================================
print("Loading data from JSONL file...")
data = []
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

df = pd.DataFrame(data)

# Ensure all expected columns exist and fill missing values with empty strings
for col in [LABEL_COL] + TEXT_FIELDS:
    if col not in df.columns:
        df[col] = ""
    else:
        df[col] = df[col].fillna("").astype(str)

# Clean out rows with empty or missing labels
df = df[df[LABEL_COL].str.strip() != ""]
df = df.dropna(subset=[LABEL_COL])


Loading data from JSONL file...


In [15]:

# ==========================================
# 3. FEATURE ENGINEERING & LABEL ENCODING
# ==========================================
# Create a unified context strategy by grouping all subjects into Text A and all bodies into Text B
df["combined_subject"] = (
    df["subject_cleaned_2"] + " " + df["tamil_subject"] + " " + df["tanglish_subject"]
)
df["combined_body"] = (
    df["body_cleaned_2"] + " " + df["tamil_body"] + " " + df["tanglish_body"]
)

# Convert categorical labels to integers
label_list = sorted(df[LABEL_COL].unique().tolist())
NUM_LABELS = len(label_list)
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

df["labels"] = df[LABEL_COL].map(label2id)

print(f"Dataset successfully loaded. Total samples: {len(df)}")
print(f"Detected {NUM_LABELS} unique intents.")


Dataset successfully loaded. Total samples: 3933
Detected 9 unique intents.


In [16]:
# ==========================================
# 4. DATASET SPLITTING
# ==========================================
label_counts = df["labels"].value_counts()
rare_labels = label_counts[label_counts < 2].index.tolist()

if rare_labels:
    print(f"Rare labels with <2 samples: {rare_labels}")
    print("Falling back to non-stratified split for this run.")

    train_df, temp_df = train_test_split(
        df, test_size=0.2, random_state=42, shuffle=True
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=42, shuffle=True
    )
else:
    train_df, temp_df = train_test_split(
        df, test_size=0.2, stratify=df["labels"], random_state=42
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, stratify=temp_df["labels"], random_state=42
    )

# Convert dataframes into Hugging Face Dataset structures
keep_cols = ["combined_subject", "combined_body", "labels"]
train_ds = Dataset.from_pandas(train_df[keep_cols].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[keep_cols].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[keep_cols].reset_index(drop=True))

Rare labels with <2 samples: [7]
Falling back to non-stratified split for this run.


In [17]:
# ==========================================
# 5. TOKENIZATION
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_function(examples):
    # Feeds pairs (Subject, Body) into the model tokenizer using XLM-R structural separators
    return tokenizer(
        examples["combined_subject"],
        examples["combined_body"],
        truncation=True,
        padding=False,  # Let DataCollator handle dynamic batch padding
        max_length=MAX_LEN,
    )


print("Tokenizing datasets...")
train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

# Format datasets explicitly for PyTorch operations
columns_to_keep = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=columns_to_keep)
val_ds.set_format(type="torch", columns=columns_to_keep)
test_ds.set_format(type="torch", columns=columns_to_keep)


Tokenizing datasets...


Map: 100%|██████████| 394/394 [00:00<00:00, 1217.88 examples/s]


In [18]:
# ==========================================
# 6. MODEL INITIALIZATION & METRICS
# ==========================================
# Load XLM-RoBERTa with appropriate classification heads
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

# Dynamic padding to adjust batch length to maximum sequence length in that batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
# ==========================================
# 7. TRAINING ARGUMENTS & EXECUTION
# ==========================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,  # Recommended LR for XLM-R fine-tuning is typically 1e-5 to 3e-5
    weight_decay=0.01,
    logging_dir=LOGS_DIR,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting XLM-RoBERTa model training...")
trainer.train()

C:\Users\vishn\AppData\Local\Temp\ipykernel_28224\521833469.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting XLM-RoBERTa model training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.792600,0.800291,0.735369,0.392682,0.386202,0.412719
2,0.586900,0.670756,0.801527,0.566448,0.628880,0.563538
3,0.432000,0.562945,0.849873,0.658222,0.647326,0.673549
4,0.257100,0.524907,0.862595,0.693367,0.688877,0.701084


d:\VishnuMK\Projects\Email\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\VishnuMK\Projects\Email\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\VishnuMK\Projects\Email\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\V

TrainOutput(global_step=1576, training_loss=0.6315882284629163, metrics={'train_runtime': 838.5817, 'train_samples_per_second': 15.006, 'train_steps_per_second': 1.879, 'total_flos': 2483398212691968.0, 'train_loss': 0.6315882284629163, 'epoch': 4.0})

In [20]:
# ==========================================
# 8. EVALUATION & SAVING
# ==========================================
print("\nEvaluating fine-tuned XLM-RoBERTa performance on Test Set...")
# Evaluate on test dataset if provided
test_results = trainer.evaluate(eval_dataset=test_ds)
print(f"Test Set Results: {test_results}")

print(f"Saving final XLM-RoBERTa model to: {OUTPUT_DIR}")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Processing complete!")


Evaluating fine-tuned XLM-RoBERTa performance on Test Set...


d:\VishnuMK\Projects\Email\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Test Set Results: {'eval_loss': 0.5327050685882568, 'eval_accuracy': 0.8756345177664975, 'eval_f1': 0.6320335788522583, 'eval_precision': 0.6575440473231301, 'eval_recall': 0.6200986297797031, 'eval_runtime': 2.2538, 'eval_samples_per_second': 174.815, 'eval_steps_per_second': 22.185, 'epoch': 4.0}
Saving final XLM-RoBERTa model to: ./xlmr_robert_base_model
Processing complete!


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "review/xlmr_robert_base_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

def predict_custom_email(subject, body, max_len=256):
    text = f"{subject} [SEP] {body}"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_len
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_id = torch.argmax(probs).item()

    return {
        "intent_type": model.config.id2label[pred_id],
        "confidence": float(probs[pred_id]),
        "all_scores": {
            model.config.id2label[i]: float(probs[i])
            for i in range(len(probs))
        }
    }

subject = "எரிக் மூனுக்கு"
body = """
எரிக் மூனுக்கு -\n\nமாற்றங்களுடன் சிட்டி பேங்க் ப்ரீபேவுக்கான கட்டமைப்பு ஸ்லைடை இணைக்கப்பட்டுள்ளது.\n\nமெலிசா சோலிஸ்\nஎன்ரான் வட அமெரிக்கா\n<கீப்டோக்கன்1>\nW: <PHONE>\nF: <PHONE>
"""

result = predict_custom_email(subject, body)
print(result)

d:\VishnuMK\Projects\Email\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The tokenizer you are loading from './xlmr_multilingual_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


{'intent_type': 'legitimate', 'confidence': 0.9984748959541321, 'all_scores': {'account_takeover': 5.7618453865870833e-05, 'credential_harvesting': 0.00011997888213954866, 'data_exfiltration': 0.0002916812081821263, 'financial_fraud': 0.00019200127280782908, 'identity_abuse': 0.00011697000445565209, 'infrastructure_exploitation': 9.345621947431937e-05, 'legitimate': 0.9984748959541321, 'legitimatef': 0.00024289156135637313, 'malware_delivery': 0.00041051366133615375}}
